# OpenPlaque RCA Ascending-Aorta Anchored Gallery v2

This replaces the previous root candidate pass, which could mistake the main pulmonary artery for the ascending aorta. This version uses DICOM patient coordinates to favor the **patient-right/anterior ascending aorta**, then searches only a thin shell around that vessel for small persistent contrast-filled branches.

There are **no sliders, widgets, clicks, or manual coordinates**. Use **Runtime → Run all**. Google Drive mounts first.


In [ ]:
# ALWAYS FIRST: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch rca-centerline-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas

import os, sys, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

SRC = Path('/content/OpenPlaque/src')
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from openplaque.study import OpenPlaqueStudy
from openplaque.rca_root_candidates import find_rca_root_candidates
print('OpenPlaque source:', SRC)


## Load source CCTA series 7


In [ ]:
ROOT = Path('/content/drive/MyDrive/OpenPlaque')
DRIVE_ZIP = ROOT / 'Full_DICOM.zip'
LOCAL_ZIP = Path('/content/Full_DICOM.zip')
EXTRACT_ROOT = '/content/full_dicom_rca_aorta_v2'
SOURCE_SERIES = 7
if not DRIVE_ZIP.exists():
    raise FileNotFoundError(f'Missing {DRIVE_ZIP}')
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    print(f'Copying Full_DICOM.zip ({DRIVE_ZIP.stat().st_size/1e9:.2f} GB)...', flush=True)
    shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)
else:
    print('Local ZIP already staged.')
shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
t=time.time()
study = OpenPlaqueStudy(str(LOCAL_ZIP), extract_root=EXTRACT_ROOT)
print(f'Scan finished in {time.time()-t:.1f}s; {len(study.series)} series found.')
source_img, source, source_files = study.load_series(SOURCE_SERIES)
spacing = source_img.GetSpacing()
print('Series 7 shape zyx:', source.shape)
print('Spacing xyz mm:', spacing)
print('Direction:', source_img.GetDirection())


## Find ascending-aorta anchored RCA root candidates


In [ ]:
print('Searching around patient-right ascending aorta...', flush=True)
t=time.time()
candidates, aorta_track, root_window = find_rca_root_candidates(source, source_img, n=8)
print(f'Finished in {time.time()-t:.1f}s. Root search window z={root_window[0]}..{root_window[1]-1}')
if not candidates:
    raise RuntimeError('No RCA root candidates found')
df = pd.DataFrame([dict(
    candidate=f'A{i+1}', z=c.z, y=c.y, x=c.x, score=c.score, HU=c.hu,
    vesselness=c.vesselness, radius_mm=c.local_radius_mm, support=c.support_slices,
    aorta_y=c.aorta_y, aorta_x=c.aorta_x, aorta_radius_mm=c.aorta_radius_mm)
    for i,c in enumerate(candidates)])
display(df)


## Verify that the large-vessel anchor is actually the ascending aorta
Each panel shows the full source slice. The black circle is the detected ascending-aorta track; the cyan point is the small-vessel candidate. Unlike the previous pass, the circle should be on the **large round vessel on the patient-right side**, not on the pulmonary trunk.


In [ ]:
OUTDIR = ROOT / 'RCA_Aorta_Anchored_v2'
OUTDIR.mkdir(parents=True, exist_ok=True)
sx, sy = float(spacing[0]), float(spacing[1])
fig, axes = plt.subplots(4,2,figsize=(12,22))
for i,(ax,c) in enumerate(zip(axes.ravel(), candidates)):
    ax.imshow(source[c.z], cmap='gray', vmin=-200, vmax=800)
    rpx = c.aorta_radius_mm / np.sqrt(sx*sy)
    ax.add_patch(Circle((c.aorta_x,c.aorta_y), rpx, fill=False, linewidth=1.4))
    ax.scatter([c.x],[c.y],s=70,facecolors='none',linewidths=1.5)
    ax.set_title(f'A{i+1}: z={c.z}  HU={c.hu:.0f}  support={c.support_slices}  r={c.local_radius_mm:.1f}mm')
    ax.axis('off')
plt.tight_layout()
p=OUTDIR/'RCA_aorta_anchor_full_context.png'
fig.savefig(p,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)
print('Saved:',p)


## Candidate-centered closeups
These closeups are the most informative cases for deciding whether a candidate is actually the RCA ostium/proximal RCA.


In [ ]:
fig, axes = plt.subplots(4,2,figsize=(12,18))
for i,(ax,c) in enumerate(zip(axes.ravel(), candidates)):
    r=75
    yc=int(round((c.y+c.aorta_y)/2)); xc=int(round((c.x+c.aorta_x)/2))
    y0,y1=max(0,yc-r),min(source.shape[1],yc+r+1)
    x0,x1=max(0,xc-r),min(source.shape[2],xc+r+1)
    ax.imshow(source[c.z,y0:y1,x0:x1],cmap='gray',vmin=-200,vmax=800)
    ax.scatter([c.x-x0],[c.y-y0],s=80,facecolors='none',linewidths=1.6)
    ax.set_title(f'A{i+1}: z={c.z}  score={c.score:.3f}  HU={c.hu:.0f}')
    ax.axis('off')
plt.tight_layout()
p=OUTDIR/'RCA_candidate_closeups.png'
fig.savefig(p,dpi=200,bbox_inches='tight')
plt.show(); plt.close(fig)
print('Saved:',p)


## Dense sequence around the strongest candidate A1
This shows whether the candidate persists as a small vessel emerging from the aortic wall over several millimeters.


In [ ]:
best=candidates[0]
dz=max(1,int(round(0.9/float(spacing[2]))))
zs=np.arange(best.z-6*dz,best.z+7*dz,dz,dtype=int)
zs=zs[(zs>=0)&(zs<source.shape[0])]
fig, axes = plt.subplots(4,4,figsize=(16,16))
for ax in axes.ravel(): ax.axis('off')
for ax,z in zip(axes.ravel(),zs):
    a=aorta_track.get(int(z),best)
    yc=int(round(a.y)); xc=int(round(a.x)); r=100
    y0,y1=max(0,yc-r),min(source.shape[1],yc+r+1)
    x0,x1=max(0,xc-r),min(source.shape[2],xc+r+1)
    ax.imshow(source[z,y0:y1,x0:x1],cmap='gray',vmin=-200,vmax=800)
    rr=a.radius_mm/np.sqrt(sx*sy)
    ax.add_patch(Circle((a.x-x0,a.y-y0),rr,fill=False,linewidth=1.0))
    ax.set_title(f'z={z}  {(z-best.z)*spacing[2]:+.1f} mm from A1')
    ax.axis('off')
plt.tight_layout()
p=OUTDIR/'RCA_A1_dense_aortic_root_sequence.png'
fig.savefig(p,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)
print('Saved:',p)
print('DONE. Upload the full-context anchor image, candidate closeups, and dense A1 sequence.')
